# Track 1 — BERT Crash-Situation Classification

This is the only file you run for Track 1. Run the cells from top to bottom.

1. Cells 1–7 load the data, fine-tune the model, and save it. This is the slow part; run it once.
2. The remaining cells reload nothing — they use the trained model in memory to produce all the results (confusion matrix, per-class metrics, Integrated Gradients).

All the reusable logic lives in `src/`; this notebook only calls it.

## 1. Setup

In [ ]:
import os
import sys

# Make the repository root importable so `src` can be found.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)
from transformers import Trainer, TrainingArguments, default_data_collator

from src.config import (
    SEED, MODEL_NAME, MAX_LENGTH, TEXT_COL, LABEL_COL,
    DATA_FOLDER, MODEL_DIR, RESULTS_DIR, set_seed,
)
from src.data_load import load_data_for_modeling
from src.split_data import stratified_train_val_test_split
from src.embedding import load_classifier_tokenizer_and_model
from src.classification.dataset import build_dataset
from src.classification.trainer import (
    compute_class_weights, make_weighted_loss, compute_metrics,
)
from src.classification.attribution import build_lig, run_class_token_analysis

os.makedirs(RESULTS_DIR, exist_ok=True)
set_seed(SEED)

## 2. Load and clean the data

In [ ]:
data = load_data_for_modeling(
    data_folder=DATA_FOLDER,
    encode_labels=True,
    verbose=True,
)

df = data["df"]
label2id = data["label2id"]
id2label = data["id2label"]
num_labels = len(label2id)

print("Number of labels:", num_labels)

## 3. Stratified train / validation / test split

In [ ]:
df_train, df_val, df_test = stratified_train_val_test_split(
    df=df,
    label_col=LABEL_COL,
    random_state=SEED,
    verbose=True,
)

## 4. Tokeniser, model, and tokenised datasets

In [ ]:
tokenizer, model = load_classifier_tokenizer_and_model(
    model_name=MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    verbose=True,
)

train_ds = build_dataset(df_train, tokenizer, TEXT_COL, LABEL_COL, MAX_LENGTH)
val_ds   = build_dataset(df_val,   tokenizer, TEXT_COL, LABEL_COL, MAX_LENGTH)
test_ds  = build_dataset(df_test,  tokenizer, TEXT_COL, LABEL_COL, MAX_LENGTH)

## 5. Fine-tune the model (slow step — run once)

In [ ]:
class_weights = compute_class_weights(df_train, LABEL_COL, num_labels)

training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=1,
    report_to="none",
)

# Class-weighted loss passed as a function, so no Trainer subclass is needed
# (requires transformers >= 4.46 for compute_loss_func).
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
    compute_loss_func=make_weighted_loss(class_weights),
)

trainer.train()
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

## 6. Predict on the test set

Builds `test_df` with the true label, predicted label, and prediction confidence. Everything below uses this frame.

In [ ]:
preds = trainer.predict(test_ds)
y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=1)

# Softmax to recover prediction confidence.
logits = preds.predictions
exp_logits = np.exp(logits - logits.max(axis=1, keepdims=True))
probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)

test_df = df_test.reset_index(drop=True).copy()
test_df["true_label_id"] = y_true
test_df["pred_label_id"] = y_pred
test_df["true_label"] = test_df["true_label_id"].map(id2label)
test_df["pred_label"] = test_df["pred_label_id"].map(id2label)
test_df["pred_conf"] = probs.max(axis=1)

print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"F1-macro: {f1_score(y_true, y_pred, average='macro'):.4f}")
test_df.to_csv(os.path.join(RESULTS_DIR, "test_predictions.csv"), index=False)

## 7. Results — confusion matrix

In [ ]:
labels_sorted = [id2label[i] for i in range(num_labels)]

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay(cm, display_labels=labels_sorted).plot(
    ax=ax, xticks_rotation=90, cmap="Blues", values_format="d"
)
plt.title("Confusion matrix — test set")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "confusion_matrix.png"), dpi=200)
plt.show()

cm_norm = confusion_matrix(y_true, y_pred, normalize="true")
fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay(cm_norm, display_labels=labels_sorted).plot(
    ax=ax, xticks_rotation=90, cmap="Blues", values_format=".2f"
)
plt.title("Normalised confusion matrix — test set")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "confusion_matrix_normalised.png"), dpi=200)
plt.show()

## 8. Results — per-class metrics and top confusions

In [ ]:
print(classification_report(y_true, y_pred, target_names=[str(l) for l in labels_sorted]))

# Most frequent off-diagonal confusions.
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)
pairs = [
    (i, j, cm_off[i, j])
    for i in range(num_labels) for j in range(num_labels)
    if cm_off[i, j] > 0
]
pairs.sort(key=lambda x: x[2], reverse=True)

print("\nTop confusion pairs:")
for i, j, count in pairs[:15]:
    print(f"True {id2label[i]} predicted as {id2label[j]}: {count}")

## 9. Results — Integrated Gradients token attribution

Builds the attribution object once, then summarises the most influential words for a class. Change `class_id` to inspect a different class.

In [ ]:
lig = build_lig(model)

subset, token_summary = run_class_token_analysis(
    df=test_df,
    class_id=6,
    model=model,
    tokenizer=tokenizer,
    lig=lig,
    id2label=id2label,
    max_length=MAX_LENGTH,
    n_examples=10,
    mode="correct",
    explain_target="pred",
    n_steps=20,
    output_dir=RESULTS_DIR,
    top_k=15,
)